# Fine-Tuning T5 for Essay Summarization

 t5-small on an Aeon essay dataset using Seq2SeqTrainer, then evaluates original vs fine‑tuned models on the test split with the same metrics and example essays.

In [ ]:
!pip install transformers[torch] datasets evaluate rouge_score pandas
!pip install accelerate bitsandbytes
!pip install bert_score
!pip install py7zr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8b03d125dde932da4c700e440a1aece5e076012ef286ae21200ab6542228ccc1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.7/142.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.9/429.9 kB 27.2 MB/s eta 0:00:00


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import Dataset, DatasetDict
import pandas as pd
import torch
import evaluate
import numpy as np
from google.colab import files

In [ ]:
print("Please upload your 'cnn_dailymail_5percent.csv' file.")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]


df_part1 = pd.read_csv(file_name)
hf_dataset = Dataset.from_pandas(df_part1)
print(hf_dataset)

Please upload your 'cnn_dailymail_5percent.csv' file.


Saving essays.csv to essays.csv
Dataset({
    features: ['title', 'description', 'essay', 'authors', 'source_url', 'thumbnail_url'],
    num_rows: 2235
})


In [ ]:
train_df = df_part1.iloc[:1600]
val_df = df_part1.iloc[1600:1800]
test_df = df_part1.iloc[1800:2235]
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")


raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

print(raw_datasets)

Train size: 1600
Validation size: 200
Test size: 435
DatasetDict({
    train: Dataset({
        features: ['title', 'description', 'essay', 'authors', 'source_url', 'thumbnail_url'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['title', 'description', 'essay', 'authors', 'source_url', 'thumbnail_url'],
        num_rows: 200
    })
    test: Dataset({
        features: ['title', 'description', 'essay', 'authors', 'source_url', 'thumbnail_url'],
        num_rows: 435
    })
})


In [ ]:
model_name_part2 = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name_part2)
prefix = "summarize: " # Same prefix as before

def preprocess_function_part2(examples):
    inputs = [prefix + doc for doc in examples["essay"]]

    model_inputs = tokenizer(inputs, max_length=1024, padding="max_length", truncation=True)
    labels = tokenizer(text_target=examples["description"], max_length=128, padding="max_length", truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function_part2, batched=True)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/435 [00:00<?, ? examples/s]

In [ ]:
### 5. Set up Trainer
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name_part2).to(device)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels (padding tokens)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    result = {key: value for key, value in result.items()}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="t5_aeon_finetuned",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="none"
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipython-input-3348223289.py:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
### 6. Train the Model
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

# Save your fine-tuned model
fine_tuned_model_path = "t5-small-finetuned-aeon"
trainer.save_model(fine_tuned_model_path)
tokenizer.save_pretrained(fine_tuned_model_path)

Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,0.930596,0.000000,0.000000,0.000000,0.000000,0.000000
2,No log,0.870741,0.000000,0.000000,0.000000,0.000000,0.000000
3,1.812200,0.855639,0.000000,0.000000,0.000000,0.000000,0.000000


Fine-tuning complete.


('t5-small-finetuned-aeon/tokenizer_config.json',
 't5-small-finetuned-aeon/special_tokens_map.json',
 't5-small-finetuned-aeon/spiece.model',
 't5-small-finetuned-aeon/added_tokens.json',
 't5-small-finetuned-aeon/tokenizer.json')

In [ ]:
print("Evaluating on the test set...")

fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(fine_tuned_model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name_part2).to(device)

models_to_compare = {
    "original_t5_small": original_model,
    "finetuned_t5_small": fine_tuned_model
}


test_references = raw_datasets['test']['description']
tokenized_test = tokenized_datasets["test"].with_format("torch")
test_dataloader = torch.utils.data.DataLoader(tokenized_test, batch_size=16)

all_test_predictions = {}

from tqdm import tqdm

for name, model_to_eval in models_to_compare.items():
    print(f"Generating test predictions for: {name}")
    predictions = []
    for batch in tqdm(test_dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model_to_eval.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=150,
            num_beams=4,
            early_stopping=True
        )
        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)
    all_test_predictions[name] = predictions

Evaluating on the test set...
Generating test predictions for: original_t5_small


100%|██████████| 28/28 [01:22<00:00,  2.93s/it]


Generating test predictions for: finetuned_t5_small


100%|██████████| 28/28 [01:37<00:00,  3.49s/it]


In [ ]:
rouge_metric = evaluate.load('rouge')
bertscore_metric = evaluate.load("bertscore")
perplexity_metric = evaluate.load("perplexity", module_type="metric")

print("\n--- Test Set Evaluation Results ---")

for name, predictions in all_test_predictions.items():
    print(f"\nResults for: {name}")

    # ROUGE
    rouge_results = rouge_metric.compute(predictions=predictions, references=test_references)
    print(f"ROUGE-1: {rouge_results['rouge1']}")
    print(f"ROUGE-2: {rouge_results['rouge2']}")

    # BERTSCORE
    print("Calculating BERTSCORE...")
    bert_results = bertscore_metric.compute(predictions=predictions, references=test_references, lang="en")
    print(f"BERTSCORE-F1 (mean): {sum(bert_results['f1']) / len(bert_results['f1'])}")


    # PERPLEXITY
    print("Calculating Perplexity...")
    valid_predictions = [p for p in predictions if p.strip()]

    if valid_predictions:
        perp_results = perplexity_metric.compute(model_id='gpt2', predictions=valid_predictions)
        print(f"Perplexity (mean, computed on {len(valid_predictions)}/{len(predictions)} valid samples): {perp_results['mean_perplexity']}")
    else:

        print("Perplexity (mean): N/A (All generated predictions were empty)")



--- Test Set Evaluation Results ---

Results for: original_t5_small
ROUGE-1: 0.13835730414419428
ROUGE-2: 0.013037566859092829
Calculating BERTSCORE...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

Perplexity (mean, computed on 435/435 valid samples): 60.96874788382958

Results for: finetuned_t5_small
ROUGE-1: 0.0002786485545106235
ROUGE-2: 0.0
Calculating BERTSCORE...


BERTSCORE-F1 (mean): 0.0019347456679947074
Calculating Perplexity...


  0%|          | 0/1 [00:00<?, ?it/s]

Perplexity (mean, computed on 1/435 valid samples): 419.2402038574219


In [ ]:
print("\n--- Qualitative Comparison (Part II) ---")

test_indices = [5, 50]
original_essays = raw_datasets['test']['essay']

for index in test_indices:
    print(f"\n--- ESSAY {index} from TEST SET ---")
    print(f"**Original Essay (snippet):**\n{original_essays[index][:500]}...\n")
    print(f"**Ground-Truth Summary:**\n{test_references[index]}\n")

    for name, predictions in all_test_predictions.items():
        generated_summary = predictions[index]
        print(f"**Summary from {name}:**\n{generated_summary}\n")

    print("-" * 30)

essay_for_part3_1 = original_essays[test_indices[0]]
essay_for_part3_2 = original_essays[test_indices[1]]


--- Qualitative Comparison (Part II) ---

--- ESSAY 5 from TEST SET ---
**Original Essay (snippet):**
When I was 12, on family vacation in New Mexico, I watched a group of elaborately-costumed Navajo men belt out one intimidating song after the next. They executed a set of beautifully coordinated dance turns to honour the four cardinal directions, each one symbolising sacred gifts from the gods. Yet the tourist-packed audience lost interest and my family, too, prepared to leave. Then, all of a sudden, the dancers were surprised by a haunting, muscled old man adorned with strange pendants, animal...

**Ground-Truth Summary:**
Religion spawns both benevolent saints and murderous fanatics. Could dopamine levels in the brain drive that switch?

**Summary from original_t5_small:**
dopamine-fuelled religion has given rise to gifted leaders and peacemakers. some of them founded not only religious traditions but also profoundly influenced the cultures and civilisations associated with those t

In [ ]:
print("--- Essay 1 for Part 3 ---")
print(essay_for_part3_1)

print("\n--- Essay 2 for Part 3 ---")
print(essay_for_part3_2)

--- Essay 1 for Part 3 ---
When I was 12, on family vacation in New Mexico, I watched a group of elaborately-costumed Navajo men belt out one intimidating song after the next. They executed a set of beautifully coordinated dance turns to honour the four cardinal directions, each one symbolising sacred gifts from the gods. Yet the tourist-packed audience lost interest and my family, too, prepared to leave. Then, all of a sudden, the dancers were surprised by a haunting, muscled old man adorned with strange pendants, animal skulls, and scars etching patterns into his body and face. Because the dancers were obviously terrified of this man I, too, became afraid and wanted to run, but we all stood rooted to the spot as he walked silently and majestically into the desert night. Afterwards, the lead dancer apologised profusely for the tribe’s shaman, or medicine man: he was holy but a bit eccentric. My 12-year-old self wondered how one might become like this extraordinary individual, so singu